In [ ]:
# pip install transformers datasets torch scikit-learn

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
import torch
import os
from sklearn.metrics import confusion_matrix, f1_score
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re
import emoji

# import train data

In [ ]:
data_path = os.getenv('DATA_PATH')
modele_path = os.getenv('MODELS')
train_data = pd.read_csv(f'{data_path}/30k_labeled_comments.csv')

In [ ]:
df = train_data[~train_data['delivery'].isin(['error', 'Error'])][['description','delivery']]
df = df.rename(columns={"description": "text", "delivery": "label"})
df["label"] = df["label"].map({
    'True':1,
    "False":0
})
ones = df[df['label']==1]
zeros = df[df['label']==0].sample(4091, random_state=42)
df = pd.concat([ones, zeros])

df.sample(3)

In [ ]:
df['label'].value_counts()

In [ ]:
def preprocessing(comment):
    """
    Preprocesses a given comment by performing the following steps:
    1. Replaces all emojis with a space.
    2. Removes URLs.
    3. Removes all non-word characters (punctuation).
    4. Removes all digits.

    Parameters:
    comment (str): The input comment to be preprocessed.

    Returns:
    str: The preprocessed comment.
    """
    comment = emoji.replace_emoji(comment, replace=" ")
    comment = re.sub(r'https?://\S+|www\.\S+', ' ', comment)
    comment = re.sub(r'[^\w\s]', ' ', comment)
    # comment = re.sub(r'\d+', ' ', comment)
    return comment
df['text'] = df['text'].apply(preprocessing)

# fintune parsbert_snappfood

In [ ]:
model_name = f"{modele_path}/HooshvareLab/sentiment-snappfood/"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

dataset = Dataset.from_pandas(df)
tokenized_datasets = dataset.map(tokenize_function, batched=True)
split = tokenized_datasets.train_test_split(test_size=0.2)
train_dataset = split["train"]
val_dataset = split["test"]

In [ ]:
num_labels = 2 
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
)
os.environ["WANDB_DISABLED"] = "true"
trainer.train()

In [ ]:
model.save_pretrained(f"{modele_path}/fine_tuned_parsbert_snappfood_for_bad_delivery_detection")
tokenizer.save_pretrained(f"{modele_path}/fine_tuned_parsbert_snappfood_for_bad_delivery_detection")

In [ ]:
# Function to compute metrics including confusion matrix
def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)  
    labels = p.label_ids  
    
    cm = confusion_matrix(labels, preds)
    
    plt.figure(figsize=(6, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["False", "True"], yticklabels=["False", "True"])
    plt.xlabel("Predicted labels")
    plt.ylabel("True labels")
    plt.title("Confusion Matrix")
    plt.show()
    
    accuracy = np.mean(preds == labels)
    return {"accuracy": accuracy, "confusion_matrix": cm}

In [ ]:
# load it if you have trained it
save_directory = f"{modele_path}/fine_tuned_parsbert_snappfood_for_bad_delivery_detection"

tokenizer = AutoTokenizer.from_pretrained(save_directory)
model = AutoModelForSequenceClassification.from_pretrained(save_directory)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

dataset = Dataset.from_pandas(df)
tokenized_datasets = dataset.map(tokenize_function, batched=True)
split = tokenized_datasets.train_test_split(test_size=0.2)
train_dataset = split["train"]
val_dataset = split["test"]

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval() 

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics  
)
trainer.evaluate()

In [ ]:
def predict(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)
    outputs = model(**inputs)
    logits = outputs.logits
    predicted_class = logits.argmax().item()
    return predicted_class, logits

new_comment = "داغون بود"
print(predict(new_comment))

# HooshvareLab/bert-fa-base-uncased

In [ ]:
# download model directly
from transformers import AutoTokenizer, AutoModelForMaskedLM

tokenizer = AutoTokenizer.from_pretrained("HooshvareLab/bert-fa-base-uncased")
model = AutoModelForMaskedLM.from_pretrained("HooshvareLab/bert-fa-base-uncased")

In [ ]:
model.save_pretrained(f"{modele_path}/HooshvareLab/bert-fa-base-uncased")
tokenizer.save_pretrained(f"{modele_path}/HooshvareLab/bert-fa-base-uncased")

In [ ]:
# load model

model_name = f"{modele_path}/HooshvareLab/bert-fa-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

dataset = Dataset.from_pandas(df)
tokenized_datasets = dataset.map(tokenize_function, batched=True)
split = tokenized_datasets.train_test_split(test_size=0.2)
train_dataset = split["train"]
val_dataset = split["test"]

In [ ]:
num_labels = 2 
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


In [ ]:
model.device

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
)
os.environ["WANDB_DISABLED"] = "true"
trainer.train()

In [ ]:
model.save_pretrained(f"{modele_path}/fine_tuned_bert-fa-base-uncased_for_bad_delivery_detection")
tokenizer.save_pretrained(f"{modele_path}/fine_tuned_bert-fa-base-uncased_for_bad_delivery_detection")

In [ ]:
def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)  
    labels = p.label_ids  
    
    cm = confusion_matrix(labels, preds)
    f1 = f1_score(labels, preds)
    
    plt.figure(figsize=(6, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["False", "True"], yticklabels=["False", "True"])
    plt.xlabel("Predicted labels")
    plt.ylabel("True labels")
    plt.title("Confusion Matrix")
    plt.show()
    
    accuracy = np.mean(preds == labels)
    return {"accuracy": accuracy, "f1_score": f1, "confusion_matrix": cm}


In [ ]:
# load it if you have trained it
save_directory = f"{modele_path}/fine_tuned_bert-fa-base-uncased_for_bad_delivery_detection"

tokenizer = AutoTokenizer.from_pretrained(save_directory)
model = AutoModelForSequenceClassification.from_pretrained(save_directory)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

dataset = Dataset.from_pandas(df)
tokenized_datasets = dataset.map(tokenize_function, batched=True)
split = tokenized_datasets.train_test_split(test_size=0.2)
train_dataset = split["train"]
val_dataset = split["test"]

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval() 

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs",
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics  
)
trainer.evaluate()

In [ ]:
def predict(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)
    outputs = model(**inputs)
    logits = outputs.logits
    predicted_class = logits.argmax().item()
    return predicted_class, logits

new_comment = "داغون بود"
print(predict(new_comment))